# 지연 리스크 설명 AI Agent v0

이 노트북은 현재 확정된 구조에 맞춘 **Notebook 검증용 구현**입니다.

구조:
1. ML 모델은 이미 `ai_prediction_results`에 지연 확률/위험 등급/예상 지연일수를 저장
2. SAFE가 아닌 prediction을 대상으로 AI Risk Analysis Agent 자동 수행
3. Context Load → Analyzer Nodes → Cause Ranking → Explanation → Validation → Persist
4. 실제 DB/LLM 연결부는 Interface로 분리

주의:
- 실제 DB 컬럼명과 JOIN 조건은 아직 확정 정보가 부족하므로 `RiskAnalysisRepository` 구현체에서 교체해야 합니다.
- LLM Provider가 확정되지 않아 현재는 `TemplateExplanationGenerator`로 동작합니다.


In [1]:
%%writefile risk_analysis_agent_v0.py

"""
risk_analysis_agent_v0.py

지연 리스크 설명 AI Agent v0
- ML 모델은 이미 ai_prediction_results에 delay_probability / risk_level / predicted_delay_days를 저장했다고 가정합니다.
- 이 모듈은 SAFE가 아닌 prediction_id를 받아 상세 원인 분석을 수행합니다.
- 실제 DB/LLM 연결부는 프로젝트 환경마다 달라지므로 Interface로 분리했습니다.

확정이 필요한 부분:
1. 실제 DB 컬럼명과 JOIN 조건
2. LLM Provider / 호출 방식
3. predicted_delay_days의 정확한 의미
4. 자재 available_qty가 예약 재고를 포함하는지 여부
5. 라인/수율/납기 기준 threshold 운영값
"""

from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
from typing import Any, Dict, Iterable, List, Optional, Protocol, Tuple
import re


# =========================
# 1. Enums & Config
# =========================

class RiskLevel(str, Enum):
    SAFE = "SAFE"
    CAUTION = "CAUTION"
    WARNING = "WARNING"
    CRITICAL = "CRITICAL"


class CauseType(str, Enum):
    MATERIAL_SHORTAGE = "MATERIAL_SHORTAGE"
    MATERIAL_DELAY = "MATERIAL_DELAY"
    LOW_YIELD = "LOW_YIELD"
    MACHINE_ABNORMAL = "MACHINE_ABNORMAL"
    LINE_ABNORMAL = "LINE_ABNORMAL"


class PriorityBand(str, Enum):
    BLOCKER = "BLOCKER"
    TIME_CRITICAL = "TIME_CRITICAL"
    FLOW_DELAY = "FLOW_DELAY"
    CONTRIBUTOR = "CONTRIBUTOR"


class Severity(str, Enum):
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"


class WorkflowStatus(str, Enum):
    SKIPPED_SAFE = "SKIPPED_SAFE"
    RUNNING = "RUNNING"
    COMPLETED = "COMPLETED"
    REVIEW_REQUIRED = "REVIEW_REQUIRED"
    FAILED = "FAILED"


@dataclass(frozen=True)
class AgentConfig:
    """
    운영 threshold는 운영팀/공정팀 기준으로 조정하세요.
    None인 값은 아직 확정되지 않은 기준이므로 해당 조건 검사를 수행하지 않습니다.
    """
    max_retry: int = 2

    # 사용자가 정의한 risk level 기준
    safe_upper: float = 0.10
    caution_upper: float = 0.40
    warning_upper: float = 0.70

    # 문서에 이미 정의된 기준 또는 일반적으로 논의된 기준
    line_utilization_threshold: float = 0.85
    throughput_min_ratio: float = 0.90
    yield_gap_major_threshold: float = 0.05

    # 아직 운영 기준 확정이 필요한 값
    queue_wait_threshold_hours: Optional[float] = None
    setup_overrun_threshold_hours: Optional[float] = None
    changeover_overrun_threshold_hours: Optional[float] = None
    due_slack_safe_threshold_hours: Optional[float] = None


CAUSE_KOREAN_NAME: Dict[CauseType, str] = {
    CauseType.MATERIAL_SHORTAGE: "자재 부족",
    CauseType.MATERIAL_DELAY: "자재 입고 지연",
    CauseType.LOW_YIELD: "수율 저하",
    CauseType.MACHINE_ABNORMAL: "설비 상태 이상",
    CauseType.LINE_ABNORMAL: "라인 상태 이상",
}


# =========================
# 2. Data Models
# =========================

@dataclass
class PredictionInput:
    prediction_id: int
    order_id: int
    plan_id: Optional[int]
    delay_probability: float
    risk_level: RiskLevel
    predicted_delay_days: Optional[float] = None
    predicted_delay_hours: Optional[float] = None
    estimated_finish_at: Optional[datetime] = None


@dataclass
class OrderInfo:
    order_id: int
    order_code: Optional[str]
    product_id: Optional[int]
    order_quantity: Optional[float]
    due_at: Optional[datetime]
    order_amount: Optional[float] = None


@dataclass
class PlanInfo:
    plan_id: Optional[int]
    order_id: int
    product_id: Optional[int]
    line_id: Optional[int]
    planned_start_at: Optional[datetime]
    planned_end_at: Optional[datetime]
    planned_qty: Optional[float]
    hourly_capacity: Optional[float] = None


@dataclass
class MaterialRequirement:
    material_id: str
    material_name: Optional[str]
    required_qty: float

    # Repository에서 "현재 생산 필요 시점 기준 가용 수량"으로 계산해서 넣는 것을 권장합니다.
    available_qty: float

    # 예약 재고 중 실제 사용할 수 있는 수량이 별도 관리된다면 입력합니다.
    usable_reserved_qty: float = 0.0

    # 입고 예정
    inbound_qty: float = 0.0
    inbound_expected_at: Optional[datetime] = None

    # 해당 주문/계획에서 이 자재가 필요한 시점
    material_required_at: Optional[datetime] = None

    # 부족으로 인해 생산이 막히는 시간을 DB/API에서 이미 계산할 수 있으면 입력합니다.
    shortage_blocking_hours: Optional[float] = None


@dataclass
class YieldStatus:
    product_id: Optional[int]
    line_id: Optional[int]
    standard_yield: Optional[float]
    recent_yield: Optional[float]
    planned_qty: Optional[float]
    hourly_production_qty: Optional[float]


@dataclass
class MachineStatus:
    machine_id: str
    line_id: Optional[int]
    status: str
    is_core_machine: bool = True
    has_alternative: bool = False
    recovery_expected_at: Optional[datetime] = None
    maintenance_start_at: Optional[datetime] = None
    maintenance_end_at: Optional[datetime] = None


@dataclass
class LineStatus:
    line_id: Optional[int]
    line_name: Optional[str] = None
    line_status: Optional[str] = None

    utilization_rate: Optional[float] = None
    queue_wait_hours: Optional[float] = None

    actual_throughput: Optional[float] = None
    standard_throughput: Optional[float] = None

    setup_time_hours: Optional[float] = None
    standard_setup_time_hours: Optional[float] = None

    changeover_time_hours: Optional[float] = None
    standard_changeover_time_hours: Optional[float] = None

    previous_process_delay_hours: Optional[float] = None
    schedule_conflict: bool = False


@dataclass
class RiskContext:
    prediction: PredictionInput
    order: OrderInfo
    plan: PlanInfo
    materials: List[MaterialRequirement] = field(default_factory=list)
    yield_status: Optional[YieldStatus] = None
    machines: List[MachineStatus] = field(default_factory=list)
    line_status: Optional[LineStatus] = None

    # 확정되지 않은 외부 계산값 또는 조회값을 담는 확장 필드
    extra: Dict[str, Any] = field(default_factory=dict)


@dataclass
class CauseCandidate:
    cause_type: CauseType
    analyzer: str
    priority_band: PriorityBand
    severity: Severity
    evidence: List[str]
    rank_reason: str
    action_hint: str
    delay_contribution_hours: Optional[float] = None
    evidence_strength: Severity = Severity.MEDIUM


@dataclass
class AnalyzerOutput:
    analyzer: str
    cause_candidates: List[CauseCandidate] = field(default_factory=list)
    evidence: List[str] = field(default_factory=list)
    risk_context: Dict[str, Any] = field(default_factory=dict)
    missing_data: List[str] = field(default_factory=list)
    errors: List[str] = field(default_factory=list)


@dataclass
class DueImpactOutput:
    due_slack_hr: Optional[float]
    predicted_delay_days: Optional[float]
    predicted_delay_hours: Optional[float]
    estimated_finish_at: Optional[datetime]
    affected_order_count: Optional[int]
    expected_loss_amount: Optional[float]
    penalty_risk: Optional[bool]
    evidence: List[str]
    missing_data: List[str] = field(default_factory=list)


@dataclass
class RankedCause:
    rank: int
    cause_type: CauseType
    korean_name: str
    priority_band: PriorityBand
    severity: Severity
    rank_reason: str
    evidence: List[str]
    action_hint: str
    delay_contribution_hours: Optional[float]


@dataclass
class CauseRankingResult:
    primary_cause_type: Optional[CauseType]
    secondary_cause_types: List[CauseType]
    selected_cause_types: List[CauseType]
    ranking_method: str
    priority_ranking: List[RankedCause]
    review_required: bool = False
    review_reason: Optional[str] = None


@dataclass
class ExplanationResult:
    cause_detail: str
    analysis_summary: str
    recommended_action: str


@dataclass
class ValidationResult:
    passed: bool
    failed_checks: List[str] = field(default_factory=list)
    warnings: List[str] = field(default_factory=list)
    retry_target: Optional[str] = None


@dataclass
class RiskAnalysisResult:
    prediction_id: int
    order_id: int
    plan_id: Optional[int]
    delay_probability: float
    risk_level: RiskLevel
    predicted_delay_days: Optional[float]

    primary_cause_type: Optional[CauseType]
    selected_cause_types: List[CauseType]

    cause_detail: str
    analysis_summary: str
    recommended_action: str

    ranking: Optional[CauseRankingResult]
    due_impact: Optional[DueImpactOutput]
    validation: Optional[ValidationResult]

    status: WorkflowStatus
    retry_count: int = 0
    error_message: Optional[str] = None


# =========================
# 3. Utility Functions
# =========================

def risk_level_from_probability(prob: float, config: AgentConfig) -> RiskLevel:
    if prob < config.safe_upper:
        return RiskLevel.SAFE
    if prob < config.caution_upper:
        return RiskLevel.CAUTION
    if prob < config.warning_upper:
        return RiskLevel.WARNING
    return RiskLevel.CRITICAL


def hours_between(later: Optional[datetime], earlier: Optional[datetime]) -> Optional[float]:
    if later is None or earlier is None:
        return None
    return (later - earlier).total_seconds() / 3600


def add_hours(dt: Optional[datetime], hours: Optional[float]) -> Optional[datetime]:
    if dt is None or hours is None:
        return None
    return dt + timedelta(hours=hours)


def fmt_qty(value: Optional[float]) -> str:
    if value is None:
        return "확인 필요"
    if abs(value - round(value)) < 1e-9:
        return f"{int(round(value)):,}"
    return f"{value:,.2f}"


def fmt_hours(value: Optional[float]) -> str:
    if value is None:
        return "확인 필요"
    if abs(value - round(value)) < 1e-9:
        return f"{int(round(value))}시간"
    return f"{value:.1f}시간"


def fmt_percent(value: Optional[float]) -> str:
    if value is None:
        return "확인 필요"
    return f"{value * 100:.1f}%"


def overlaps(start_a: Optional[datetime], end_a: Optional[datetime],
             start_b: Optional[datetime], end_b: Optional[datetime]) -> bool:
    if start_a is None or end_a is None or start_b is None or end_b is None:
        return False
    return max(start_a, start_b) < min(end_a, end_b)


# =========================
# 4. Repository Interface
# =========================

class RiskAnalysisRepository(Protocol):
    """
    실제 구현에서는 SQLAlchemy, psycopg, Spring API Client 등으로 교체하세요.
    """

    def fetch_non_safe_predictions(self, limit: int = 100) -> List[PredictionInput]:
        ...

    def get_prediction(self, prediction_id: int) -> PredictionInput:
        ...

    def load_context(self, prediction_id: int) -> RiskContext:
        ...

    def persist_analysis(self, result: RiskAnalysisResult) -> None:
        """
        저장 정책:
        - ai_prediction_results.cause_detail 갱신
        - ai_prediction_results.analysis_summary 갱신
        - ai_prediction_results.recommended_action 갱신
        - ai_prediction_causes에 selected_cause_types 다건 저장
        """
        ...

    def mark_review_required(self, prediction_id: int, reason: str) -> None:
        ...


# =========================
# 5. Analyzer Nodes
# =========================

class DueImpactAnalyzer:
    def analyze(self, ctx: RiskContext) -> DueImpactOutput:
        prediction = ctx.prediction
        order = ctx.order
        plan = ctx.plan

        missing: List[str] = []

        predicted_delay_hours = prediction.predicted_delay_hours
        if predicted_delay_hours is None and prediction.predicted_delay_days is not None:
            predicted_delay_hours = prediction.predicted_delay_days * 24

        # 우선순위:
        # 1) ML 또는 upstream에서 estimated_finish_at 제공
        # 2) 없으면 planned_end_at + predicted_delay_hours 사용
        # 이 fallback의 타당성은 프로젝트에서 확인 필요합니다.
        estimated_finish_at = prediction.estimated_finish_at
        if estimated_finish_at is None and plan.planned_end_at is not None and predicted_delay_hours is not None:
            estimated_finish_at = add_hours(plan.planned_end_at, predicted_delay_hours)

        if order.due_at is None:
            missing.append("order.due_at")
        if estimated_finish_at is None:
            missing.append("estimated_finish_at 또는 plan.planned_end_at + predicted_delay_hours")

        due_slack_hr = hours_between(order.due_at, estimated_finish_at) if order.due_at and estimated_finish_at else None

        affected_order_count = ctx.extra.get("affected_order_count")
        expected_loss_amount = ctx.extra.get("expected_loss_amount")
        penalty_risk = ctx.extra.get("penalty_risk")

        evidence: List[str] = []
        if due_slack_hr is not None:
            if due_slack_hr >= 0:
                evidence.append(f"납기 여유 시간은 {fmt_hours(due_slack_hr)}입니다.")
            else:
                evidence.append(f"예상 완료 시각이 납기보다 {fmt_hours(abs(due_slack_hr))} 늦습니다.")
        if prediction.predicted_delay_days is not None:
            evidence.append(f"ML 모델의 예상 지연 일수는 {prediction.predicted_delay_days:.2f}일입니다.")
        if affected_order_count is not None:
            evidence.append(f"후속 영향 주문 수는 {affected_order_count}건입니다.")
        if expected_loss_amount is not None:
            evidence.append(f"예상 손실 금액은 {fmt_qty(float(expected_loss_amount))}원입니다.")

        return DueImpactOutput(
            due_slack_hr=due_slack_hr,
            predicted_delay_days=prediction.predicted_delay_days,
            predicted_delay_hours=predicted_delay_hours,
            estimated_finish_at=estimated_finish_at,
            affected_order_count=affected_order_count,
            expected_loss_amount=expected_loss_amount,
            penalty_risk=penalty_risk,
            evidence=evidence,
            missing_data=missing,
        )


class MaterialAnalyzer:
    def analyze(self, ctx: RiskContext, due: DueImpactOutput) -> AnalyzerOutput:
        out = AnalyzerOutput(analyzer="MaterialAnalyzer")

        if not ctx.materials:
            out.missing_data.append("materials")
            return out

        for m in ctx.materials:
            if m.material_required_at is None:
                out.missing_data.append(f"{m.material_id}.material_required_at")
                continue

            available_now = m.available_qty + m.usable_reserved_qty
            inbound_on_time = (
                m.inbound_qty
                if m.inbound_expected_at is not None and m.inbound_expected_at <= m.material_required_at
                else 0.0
            )
            available_at_need = available_now + inbound_on_time
            total_eventual = available_now + m.inbound_qty

            shortage_at_need = max(0.0, m.required_qty - available_at_need)
            shortage_ratio = shortage_at_need / m.required_qty if m.required_qty > 0 else 0.0

            inbound_delay_hours = hours_between(m.inbound_expected_at, m.material_required_at)

            # MATERIAL_DELAY: 입고 예정으로 부족분을 충족할 수 있지만 필요한 시점보다 늦음
            if (
                shortage_at_need > 0
                and m.inbound_qty > 0
                and total_eventual >= m.required_qty
                and inbound_delay_hours is not None
                and inbound_delay_hours > 0
            ):
                severity = Severity.HIGH if due.due_slack_hr is not None and inbound_delay_hours >= max(due.due_slack_hr, 0) else Severity.MEDIUM
                evidence = [
                    f"{m.material_name or m.material_id}의 입고 예정 시각이 생산 필요 시점보다 {fmt_hours(inbound_delay_hours)} 늦습니다.",
                    f"필요 수량 {fmt_qty(m.required_qty)} 대비 현재 가용 수량은 {fmt_qty(available_now)}입니다.",
                ]
                out.cause_candidates.append(
                    CauseCandidate(
                        cause_type=CauseType.MATERIAL_DELAY,
                        analyzer=out.analyzer,
                        priority_band=PriorityBand.TIME_CRITICAL,
                        severity=severity,
                        evidence=evidence,
                        rank_reason="자재 입고 예정 시점이 생산 필요 시점보다 늦어 계획된 생산 시작이 지연될 수 있음",
                        action_hint=f"{m.material_name or m.material_id} 입고 일정 앞당김 가능 여부 확인",
                        delay_contribution_hours=inbound_delay_hours,
                        evidence_strength=Severity.HIGH,
                    )
                )

            # MATERIAL_SHORTAGE: 입고까지 고려해도 필요 수량 충족 불가 또는 입고 정보 없음
            if shortage_at_need > 0 and total_eventual < m.required_qty and m.inbound_qty > 0:
                eventual_shortage = max(0.0, m.required_qty - total_eventual)
                evidence = [
                    f"{m.material_name or m.material_id} 필요 수량은 {fmt_qty(m.required_qty)}이나 생산 필요 시점 기준 가용 수량은 {fmt_qty(available_at_need)}입니다.",
                    f"부족 수량은 {fmt_qty(shortage_at_need)}이며 부족 비율은 {fmt_percent(shortage_ratio)}입니다.",
                ]
                if m.inbound_qty > 0:
                    evidence.append(f"입고 예정 수량까지 반영해도 {fmt_qty(eventual_shortage)}이 부족합니다.")

                out.cause_candidates.append(
                    CauseCandidate(
                        cause_type=CauseType.MATERIAL_SHORTAGE,
                        analyzer=out.analyzer,
                        priority_band=PriorityBand.BLOCKER,
                        severity=Severity.HIGH,
                        evidence=evidence,
                        rank_reason="필수 자재의 가용 재고가 필요 수량보다 부족하여 생산 착수 자체가 지연될 가능성이 높음",
                        action_hint=f"{m.material_name or m.material_id} 긴급 입고 또는 대체 자재 사용 가능 여부 확인",
                        delay_contribution_hours=m.shortage_blocking_hours,
                        evidence_strength=Severity.HIGH,
                    )
                )

            # 입고가 아예 없고 현재 부족한 경우
            if shortage_at_need > 0 and m.inbound_qty <= 0:
                evidence = [
                    f"{m.material_name or m.material_id} 필요 수량은 {fmt_qty(m.required_qty)}이나 현재 가용 수량은 {fmt_qty(available_now)}입니다.",
                    f"부족 수량은 {fmt_qty(shortage_at_need)}이며 부족 비율은 {fmt_percent(shortage_ratio)}입니다.",
                    "등록된 입고 예정 수량이 없습니다.",
                ]
                out.cause_candidates.append(
                    CauseCandidate(
                        cause_type=CauseType.MATERIAL_SHORTAGE,
                        analyzer=out.analyzer,
                        priority_band=PriorityBand.BLOCKER,
                        severity=Severity.HIGH,
                        evidence=evidence,
                        rank_reason="필수 자재가 부족하고 입고 예정 정보가 없어 생산 착수 자체가 지연될 수 있음",
                        action_hint=f"{m.material_name or m.material_id} 구매 요청 또는 대체 자재 검토",
                        delay_contribution_hours=m.shortage_blocking_hours,
                        evidence_strength=Severity.HIGH,
                    )
                )

        return out


class YieldAnalyzer:
    def __init__(self, config: AgentConfig):
        self.config = config

    def analyze(self, ctx: RiskContext, due: DueImpactOutput) -> AnalyzerOutput:
        out = AnalyzerOutput(analyzer="YieldAnalyzer")
        y = ctx.yield_status

        if y is None:
            out.missing_data.append("yield_status")
            return out

        required = ["standard_yield", "recent_yield", "planned_qty"]
        for name in required:
            if getattr(y, name) is None:
                out.missing_data.append(f"yield_status.{name}")

        if out.missing_data:
            return out

        assert y.standard_yield is not None
        assert y.recent_yield is not None
        assert y.planned_qty is not None

        if y.recent_yield <= 0:
            out.errors.append("recent_yield must be greater than 0")
            return out

        if y.recent_yield < y.standard_yield:
            yield_gap = y.standard_yield - y.recent_yield

            # 목표 양품 생산량을 맞추기 위해 추가 투입/생산이 필요한 양의 근사치입니다.
            # 실제 공식이 별도로 있다면 이 부분을 교체하세요.
            additional_required_qty = max(0.0, y.planned_qty * (y.standard_yield / y.recent_yield - 1))
            additional_hours = None
            if y.hourly_production_qty is not None and y.hourly_production_qty > 0:
                additional_hours = additional_required_qty / y.hourly_production_qty

            band = PriorityBand.CONTRIBUTOR
            severity = Severity.MEDIUM
            if additional_hours is not None and due.due_slack_hr is not None and additional_hours >= max(due.due_slack_hr, 0):
                band = PriorityBand.TIME_CRITICAL
                severity = Severity.HIGH
            elif yield_gap >= self.config.yield_gap_major_threshold:
                severity = Severity.MEDIUM

            evidence = [
                f"최근 수율은 {fmt_percent(y.recent_yield)}로 기준 수율 {fmt_percent(y.standard_yield)}보다 낮습니다.",
                f"수율 차이는 {fmt_percent(yield_gap)}입니다.",
                f"목표 수량 달성을 위해 약 {fmt_qty(additional_required_qty)}의 추가 생산이 필요합니다.",
            ]
            if additional_hours is not None:
                evidence.append(f"추가 생산 예상 시간은 약 {fmt_hours(additional_hours)}입니다.")

            out.cause_candidates.append(
                CauseCandidate(
                    cause_type=CauseType.LOW_YIELD,
                    analyzer=out.analyzer,
                    priority_band=band,
                    severity=severity,
                    evidence=evidence,
                    rank_reason="기준 수율 대비 최근 수율이 낮아 목표 생산량 달성을 위해 추가 생산 시간이 필요함",
                    action_hint="수율 저하 원인 확인 및 추가 생산 가능 시간 확보",
                    delay_contribution_hours=additional_hours,
                    evidence_strength=Severity.MEDIUM,
                )
            )

        return out


class MachineAnalyzer:
    ABNORMAL_STATUSES = {"ERROR", "STOPPED", "MAINTENANCE"}

    def analyze(self, ctx: RiskContext, due: DueImpactOutput) -> AnalyzerOutput:
        out = AnalyzerOutput(analyzer="MachineAnalyzer")
        if not ctx.machines:
            out.missing_data.append("machines")
            return out

        plan = ctx.plan

        for machine in ctx.machines:
            status = machine.status.upper()

            if status not in self.ABNORMAL_STATUSES:
                continue

            if machine.line_id is not None and plan.line_id is not None and machine.line_id != plan.line_id:
                continue

            if status == "MAINTENANCE":
                # MAINTENANCE가 계획 시간과 겹치지 않으면 지연 원인에서 제외합니다.
                if not overlaps(
                    machine.maintenance_start_at,
                    machine.maintenance_end_at,
                    plan.planned_start_at,
                    plan.planned_end_at,
                ):
                    continue

            delay_hours = hours_between(machine.recovery_expected_at, plan.planned_start_at)

            band = PriorityBand.BLOCKER
            severity = Severity.HIGH if status in {"ERROR", "STOPPED"} else Severity.MEDIUM
            if status == "MAINTENANCE":
                band = PriorityBand.CONTRIBUTOR

            evidence = [
                f"{machine.machine_id} 상태가 {status}입니다.",
            ]
            if machine.is_core_machine:
                evidence.append("해당 설비는 배정 라인의 핵심 설비입니다.")
            if machine.recovery_expected_at and plan.planned_start_at and delay_hours is not None and delay_hours > 0:
                evidence.append(f"복구 예상 시각이 생산 시작 예정 시각보다 {fmt_hours(delay_hours)} 늦습니다.")
            if machine.has_alternative:
                evidence.append("대체 설비 또는 대체 라인이 존재합니다.")
            else:
                evidence.append("대체 설비 또는 대체 라인 정보가 없습니다.")

            out.cause_candidates.append(
                CauseCandidate(
                    cause_type=CauseType.MACHINE_ABNORMAL,
                    analyzer=out.analyzer,
                    priority_band=band,
                    severity=severity,
                    evidence=evidence,
                    rank_reason="배정 라인의 핵심 설비가 비정상 상태로 생산 진행 자체가 제한됨",
                    action_hint=f"{machine.machine_id} 복구 일정 확인 또는 대체 설비/라인 검토",
                    delay_contribution_hours=delay_hours if delay_hours is not None and delay_hours > 0 else None,
                    evidence_strength=Severity.HIGH,
                )
            )

        return out


class LineProcessAnalyzer:
    def __init__(self, config: AgentConfig):
        self.config = config

    def analyze(self, ctx: RiskContext, due: DueImpactOutput) -> AnalyzerOutput:
        out = AnalyzerOutput(analyzer="LineProcessAnalyzer")
        line = ctx.line_status
        plan = ctx.plan

        if line is None:
            out.missing_data.append("line_status")
            return out

        evidence: List[str] = []
        delay_parts: List[float] = []

        if line.utilization_rate is not None and line.utilization_rate >= self.config.line_utilization_threshold:
            evidence.append(
                f"{line.line_name or line.line_id}의 현재 가동률이 {fmt_percent(line.utilization_rate)}로 기준 {fmt_percent(self.config.line_utilization_threshold)}를 초과합니다."
            )

        if (
            self.config.queue_wait_threshold_hours is not None
            and line.queue_wait_hours is not None
            and line.queue_wait_hours >= self.config.queue_wait_threshold_hours
        ):
            evidence.append(f"라인 대기시간이 {fmt_hours(line.queue_wait_hours)}로 기준을 초과합니다.")
            delay_parts.append(line.queue_wait_hours)
        elif line.queue_wait_hours is not None and line.queue_wait_hours > 0:
            evidence.append(f"라인 대기시간은 {fmt_hours(line.queue_wait_hours)}입니다.")
            delay_parts.append(line.queue_wait_hours)

        if (
            line.actual_throughput is not None
            and line.standard_throughput is not None
            and line.standard_throughput > 0
            and line.actual_throughput < line.standard_throughput * self.config.throughput_min_ratio
        ):
            evidence.append(
                f"실제 처리량 {fmt_qty(line.actual_throughput)}이 기준 처리량 {fmt_qty(line.standard_throughput)}의 {fmt_percent(self.config.throughput_min_ratio)} 미만입니다."
            )
            if plan.planned_qty is not None and line.actual_throughput > 0:
                standard_hours = plan.planned_qty / line.standard_throughput
                actual_hours = plan.planned_qty / line.actual_throughput
                delay_parts.append(max(0.0, actual_hours - standard_hours))

        if (
            line.setup_time_hours is not None
            and line.standard_setup_time_hours is not None
            and line.setup_time_hours > line.standard_setup_time_hours
        ):
            overrun = line.setup_time_hours - line.standard_setup_time_hours
            evidence.append(
                f"SETUP 시간이 표준 {fmt_hours(line.standard_setup_time_hours)} 대비 {fmt_hours(line.setup_time_hours)}으로 증가했습니다."
            )
            delay_parts.append(overrun)

        if (
            line.changeover_time_hours is not None
            and line.standard_changeover_time_hours is not None
            and line.changeover_time_hours > line.standard_changeover_time_hours
        ):
            overrun = line.changeover_time_hours - line.standard_changeover_time_hours
            evidence.append(
                f"제품 전환 시간이 표준 {fmt_hours(line.standard_changeover_time_hours)} 대비 {fmt_hours(line.changeover_time_hours)}으로 증가했습니다."
            )
            delay_parts.append(overrun)

        if line.previous_process_delay_hours is not None and line.previous_process_delay_hours > 0:
            evidence.append(f"전공정 지연으로 후공정 투입이 {fmt_hours(line.previous_process_delay_hours)} 지연됩니다.")
            delay_parts.append(line.previous_process_delay_hours)

        if line.schedule_conflict:
            evidence.append("생산 순서 또는 일정 충돌이 확인되었습니다.")

        abnormal_line_statuses = {"FAULT", "SETUP", "DELAYED", "BLOCKED"}
        if line.line_status and line.line_status.upper() in abnormal_line_statuses:
            evidence.append(f"라인 상태가 {line.line_status}입니다.")

        if not evidence:
            return out

        delay_contribution = sum(delay_parts) if delay_parts else None

        band = PriorityBand.FLOW_DELAY
        severity = Severity.MEDIUM

        if delay_contribution is not None:
            due_slack_positive = max(due.due_slack_hr or 0.0, 0.0)
            predicted_delay_hours = due.predicted_delay_hours
            if due_slack_positive > 0 and delay_contribution >= due_slack_positive:
                band = PriorityBand.TIME_CRITICAL
                severity = Severity.HIGH
            elif predicted_delay_hours is not None and predicted_delay_hours > 0 and delay_contribution >= predicted_delay_hours * 0.5:
                band = PriorityBand.TIME_CRITICAL
                severity = Severity.HIGH

        out.cause_candidates.append(
            CauseCandidate(
                cause_type=CauseType.LINE_ABNORMAL,
                analyzer=out.analyzer,
                priority_band=band,
                severity=severity,
                evidence=evidence,
                rank_reason="라인 대기시간, 처리량, SETUP 또는 공정 흐름 문제로 생산 시작 및 완료 시점이 지연될 가능성이 큼",
                action_hint="대체 라인 배정, 생산 순서 조정 또는 SETUP 시간 단축 가능성 검토",
                delay_contribution_hours=delay_contribution,
                evidence_strength=Severity.MEDIUM,
            )
        )

        return out


# =========================
# 6. Cause Ranking Node
# =========================

class CauseRankingNode:
    def __init__(self):
        self.band_priority = {
            PriorityBand.BLOCKER: 1,
            PriorityBand.TIME_CRITICAL: 2,
            PriorityBand.FLOW_DELAY: 3,
            PriorityBand.CONTRIBUTOR: 4,
        }
        # 같은 band, delay contribution도 비슷할 때 사용하는 최종 tie-breaker입니다.
        self.cause_type_priority = {
            CauseType.MACHINE_ABNORMAL: 1,
            CauseType.MATERIAL_SHORTAGE: 2,
            CauseType.MATERIAL_DELAY: 3,
            CauseType.LINE_ABNORMAL: 4,
            CauseType.LOW_YIELD: 5,
        }

    def rank(self, analyzer_outputs: List[AnalyzerOutput], due: DueImpactOutput) -> CauseRankingResult:
        candidates: List[CauseCandidate] = [
            c for output in analyzer_outputs for c in output.cause_candidates
        ]

        if not candidates:
            return CauseRankingResult(
                primary_cause_type=None,
                secondary_cause_types=[],
                selected_cause_types=[],
                ranking_method="RULE_BASED_WITH_DELAY_CONTRIBUTION_TIE_BREAKER",
                priority_ranking=[],
                review_required=True,
                review_reason="원인 후보가 생성되지 않았습니다. 데이터 부족 또는 규칙 미충족 가능성이 있습니다.",
            )

        # 납기 여유 시간을 초과하는 지연 기여 원인은 TIME_CRITICAL로 승격합니다.
        promoted: List[CauseCandidate] = []
        due_slack_positive = max(due.due_slack_hr or 0.0, 0.0)
        for c in candidates:
            if (
                c.priority_band not in {PriorityBand.BLOCKER, PriorityBand.TIME_CRITICAL}
                and c.delay_contribution_hours is not None
                and due_slack_positive > 0
                and c.delay_contribution_hours >= due_slack_positive
            ):
                promoted.append(
                    CauseCandidate(
                        cause_type=c.cause_type,
                        analyzer=c.analyzer,
                        priority_band=PriorityBand.TIME_CRITICAL,
                        severity=Severity.HIGH,
                        evidence=c.evidence,
                        rank_reason=c.rank_reason + " 납기 여유 시간을 초과하는 지연 기여가 있어 우선순위를 승격했습니다.",
                        action_hint=c.action_hint,
                        delay_contribution_hours=c.delay_contribution_hours,
                        evidence_strength=c.evidence_strength,
                    )
                )
            else:
                promoted.append(c)

        def sort_key(c: CauseCandidate) -> Tuple[int, float, int, int]:
            delay = c.delay_contribution_hours if c.delay_contribution_hours is not None else -1.0
            return (
                self.band_priority[c.priority_band],
                -delay,
                -len(c.evidence),
                self.cause_type_priority[c.cause_type],
            )

        promoted.sort(key=sort_key)

        # 같은 cause_type이 여러 Analyzer/여러 자재에서 반복될 수 있으므로 DB 저장 기준에 맞게 원인 타입 단위로 병합합니다.
        merged: Dict[CauseType, CauseCandidate] = {}
        for c in promoted:
            if c.cause_type not in merged:
                merged[c.cause_type] = c
                continue

            prev = merged[c.cause_type]
            evidence = list(prev.evidence)
            for ev in c.evidence:
                if ev not in evidence:
                    evidence.append(ev)

            # 같은 cause_type 안에서는 더 큰 지연 기여 시간을 대표값으로 둡니다.
            prev_delay = prev.delay_contribution_hours
            cur_delay = c.delay_contribution_hours
            if prev_delay is None:
                delay = cur_delay
            elif cur_delay is None:
                delay = prev_delay
            else:
                delay = max(prev_delay, cur_delay)

            # 이미 정렬된 상태이므로 priority_band/severity/rank_reason/action_hint는 기존 대표 후보를 유지합니다.
            merged[c.cause_type] = CauseCandidate(
                cause_type=prev.cause_type,
                analyzer=prev.analyzer,
                priority_band=prev.priority_band,
                severity=prev.severity,
                evidence=evidence,
                rank_reason=prev.rank_reason,
                action_hint=prev.action_hint,
                delay_contribution_hours=delay,
                evidence_strength=prev.evidence_strength,
            )

        merged_candidates = list(merged.values())
        merged_candidates.sort(key=sort_key)

        ranked = [
            RankedCause(
                rank=i + 1,
                cause_type=c.cause_type,
                korean_name=CAUSE_KOREAN_NAME[c.cause_type],
                priority_band=c.priority_band,
                severity=c.severity,
                rank_reason=c.rank_reason,
                evidence=c.evidence,
                action_hint=c.action_hint,
                delay_contribution_hours=c.delay_contribution_hours,
            )
            for i, c in enumerate(merged_candidates)
        ]

        selected = [r.cause_type for r in ranked]

        return CauseRankingResult(
            primary_cause_type=ranked[0].cause_type if ranked else None,
            secondary_cause_types=selected[1:],
            selected_cause_types=selected,
            ranking_method="RULE_BASED_WITH_DELAY_CONTRIBUTION_TIE_BREAKER",
            priority_ranking=ranked,
            review_required=False,
        )


# =========================
# 7. Explanation LLM Node
# =========================

class ExplanationGenerator(Protocol):
    def generate(self, ctx: RiskContext, due: DueImpactOutput, ranking: CauseRankingResult) -> ExplanationResult:
        ...


class TemplateExplanationGenerator:
    """
    Notebook 검증용 deterministic generator입니다.
    실제 운영에서는 이 클래스를 OpenAI/Azure/Bedrock/사내 LLM client로 교체하세요.

    중요한 원칙:
    - 원인 판단은 하지 않습니다.
    - CauseRankingNode가 확정한 순위와 evidence만 사용합니다.
    """

    def generate(self, ctx: RiskContext, due: DueImpactOutput, ranking: CauseRankingResult) -> ExplanationResult:
        prediction = ctx.prediction
        order_code = ctx.order.order_code or str(ctx.order.order_id)

        if ranking.review_required or not ranking.priority_ranking:
            return ExplanationResult(
                cause_detail=(
                    f"주문 {order_code}은 지연 위험이 있으나, 현재 데이터만으로 주요 원인을 확정하기 어렵습니다. "
                    "자재, 라인, 설비, 수율 데이터의 누락 여부를 확인해야 합니다."
                ),
                analysis_summary="지연 원인 확인 필요",
                recommended_action="누락 데이터 확인 후 리스크 상세 분석을 재실행해야 합니다.",
            )

        primary = ranking.priority_ranking[0]
        secondary = ranking.priority_ranking[1:]

        primary_evidence = " ".join(primary.evidence[:3])
        secondary_text = ""
        if secondary:
            sec_names = ", ".join([s.korean_name for s in secondary[:2]])
            sec_evidence = " ".join([e for s in secondary[:2] for e in s.evidence[:1]])
            secondary_text = f" 보조 원인으로는 {sec_names}가 확인됩니다. {sec_evidence}"

        due_text = ""
        if due.due_slack_hr is not None:
            if due.due_slack_hr >= 0:
                due_text = f" 납기 여유 시간은 {fmt_hours(due.due_slack_hr)}입니다."
            else:
                due_text = f" 예상 완료 시각이 납기보다 {fmt_hours(abs(due.due_slack_hr))} 늦습니다."

        cause_detail = (
            f"주요 원인은 {primary.korean_name}입니다. "
            f"{primary.rank_reason} {primary_evidence}"
            f"{secondary_text}{due_text}"
        ).strip()

        secondary_names = [s.korean_name for s in secondary]
        if secondary_names:
            analysis_summary = f"주요 원인은 {primary.korean_name}이며, 보조 원인은 {', '.join(secondary_names)}입니다."
        else:
            analysis_summary = f"주요 원인은 {primary.korean_name}입니다."

        action_hints = []
        for r in ranking.priority_ranking:
            if r.action_hint not in action_hints:
                action_hints.append(r.action_hint)

        if due.penalty_risk:
            action_hints.append("위약 가능성이 있어 고객 대응 및 납기 조정 가능성을 검토해야 합니다.")

        recommended_action = " ".join([f"{i+1}. {a}" for i, a in enumerate(action_hints[:4])])

        return ExplanationResult(
            cause_detail=cause_detail,
            analysis_summary=analysis_summary,
            recommended_action=recommended_action,
        )


# =========================
# 8. Validation Node
# =========================

class ValidationNode:
    def __init__(self, config: AgentConfig):
        self.config = config

    def validate(
        self,
        ctx: RiskContext,
        due: DueImpactOutput,
        ranking: CauseRankingResult,
        explanation: ExplanationResult,
    ) -> ValidationResult:
        failed: List[str] = []
        warnings: List[str] = []

        if not explanation.cause_detail.strip():
            failed.append("cause_detail이 비어 있습니다.")
        if not explanation.analysis_summary.strip():
            failed.append("analysis_summary가 비어 있습니다.")
        if not explanation.recommended_action.strip():
            failed.append("recommended_action이 비어 있습니다.")

        allowed = set(CauseType)
        for c in ranking.selected_cause_types:
            if c not in allowed:
                failed.append(f"허용되지 않은 cause_type입니다: {c}")

        expected_risk_level = risk_level_from_probability(ctx.prediction.delay_probability, self.config)
        if ctx.prediction.risk_level != expected_risk_level:
            failed.append(
                f"risk_level이 delay_probability 기준과 불일치합니다. "
                f"expected={expected_risk_level.value}, actual={ctx.prediction.risk_level.value}"
            )

        if ctx.prediction.risk_level != RiskLevel.SAFE and len(ranking.selected_cause_types) == 0:
            failed.append("SAFE가 아닌데 ai_prediction_causes에 저장할 cause_type이 없습니다.")

        # 간단한 숫자 hallucination 검사:
        # explanation에 등장한 숫자가 evidence/prediction/due text에 존재하지 않으면 warning 처리합니다.
        # 운영에서는 더 정교한 fact-checker로 교체하는 것을 권장합니다.
        allowed_text_parts = []
        for r in ranking.priority_ranking:
            allowed_text_parts.extend(r.evidence)
        allowed_text_parts.extend(due.evidence)
        allowed_text_parts.append(str(ctx.prediction.delay_probability))
        if ctx.prediction.predicted_delay_days is not None:
            allowed_text_parts.append(str(ctx.prediction.predicted_delay_days))

        allowed_numbers = self._extract_numbers(" ".join(allowed_text_parts))
        output_numbers = self._extract_numbers(
            " ".join([explanation.cause_detail, explanation.analysis_summary, explanation.recommended_action])
        )
        unexpected_numbers = sorted(output_numbers - allowed_numbers)
        if unexpected_numbers:
            warnings.append(f"근거에 없는 숫자가 설명에 포함되어 있을 수 있습니다: {unexpected_numbers}")

        retry_target = None
        if failed:
            if any("cause_detail" in f or "analysis_summary" in f or "recommended_action" in f for f in failed):
                retry_target = "ExplanationLLM"
            else:
                retry_target = "AnalyzerOrRanking"

        return ValidationResult(
            passed=not failed,
            failed_checks=failed,
            warnings=warnings,
            retry_target=retry_target,
        )

    @staticmethod
    def _extract_numbers(text: str) -> set[str]:
        return set(re.findall(r"\d+(?:\.\d+)?", text))


# =========================
# 9. Persist Node
# =========================

class PersistNode:
    def __init__(self, repository: RiskAnalysisRepository):
        self.repository = repository

    def persist(self, result: RiskAnalysisResult) -> None:
        self.repository.persist_analysis(result)


# =========================
# 10. Workflow Controller
# =========================

class WorkflowController:
    def __init__(
        self,
        repository: RiskAnalysisRepository,
        config: Optional[AgentConfig] = None,
        explanation_generator: Optional[ExplanationGenerator] = None,
    ):
        self.repository = repository
        self.config = config or AgentConfig()

        self.due_analyzer = DueImpactAnalyzer()
        self.analyzers = [
            MaterialAnalyzer(),
            YieldAnalyzer(self.config),
            MachineAnalyzer(),
            LineProcessAnalyzer(self.config),
        ]
        self.ranking_node = CauseRankingNode()
        self.explanation_generator = explanation_generator or TemplateExplanationGenerator()
        self.validation_node = ValidationNode(self.config)
        self.persist_node = PersistNode(repository)

    def run_pending_non_safe(self, limit: int = 100) -> List[RiskAnalysisResult]:
        predictions = self.repository.fetch_non_safe_predictions(limit=limit)
        results = []
        for p in predictions:
            results.append(self.run_for_prediction(p.prediction_id))
        return results

    def run_for_prediction(self, prediction_id: int) -> RiskAnalysisResult:
        retry_count = 0

        prediction = self.repository.get_prediction(prediction_id)
        expected_level = risk_level_from_probability(prediction.delay_probability, self.config)

        if prediction.risk_level == RiskLevel.SAFE or expected_level == RiskLevel.SAFE:
            return RiskAnalysisResult(
                prediction_id=prediction.prediction_id,
                order_id=prediction.order_id,
                plan_id=prediction.plan_id,
                delay_probability=prediction.delay_probability,
                risk_level=prediction.risk_level,
                predicted_delay_days=prediction.predicted_delay_days,
                primary_cause_type=None,
                selected_cause_types=[],
                cause_detail="",
                analysis_summary="SAFE 등급으로 상세 원인 분석을 수행하지 않았습니다.",
                recommended_action="",
                ranking=None,
                due_impact=None,
                validation=None,
                status=WorkflowStatus.SKIPPED_SAFE,
            )

        try:
            ctx = self._retry("ContextLoad", lambda: self.repository.load_context(prediction_id))
            due = self._retry("DueImpactAnalyzer", lambda: self.due_analyzer.analyze(ctx))

            analyzer_outputs = []
            for analyzer in self.analyzers:
                output = self._retry(analyzer.__class__.__name__, lambda a=analyzer: a.analyze(ctx, due))
                analyzer_outputs.append(output)

            ranking = self._retry("CauseRankingNode", lambda: self.ranking_node.rank(analyzer_outputs, due))

            explanation = None
            validation = None
            for attempt in range(self.config.max_retry + 1):
                retry_count = attempt
                explanation = self._retry(
                    "ExplanationLLM",
                    lambda: self.explanation_generator.generate(ctx, due, ranking),
                )
                validation = self.validation_node.validate(ctx, due, ranking, explanation)
                if validation.passed:
                    break
                if validation.retry_target != "ExplanationLLM":
                    break

            assert explanation is not None
            assert validation is not None

            if not validation.passed or ranking.review_required:
                reason = "; ".join(validation.failed_checks) if validation.failed_checks else ranking.review_reason or "검토 필요"
                self.repository.mark_review_required(prediction_id, reason)
                return RiskAnalysisResult(
                    prediction_id=ctx.prediction.prediction_id,
                    order_id=ctx.prediction.order_id,
                    plan_id=ctx.prediction.plan_id,
                    delay_probability=ctx.prediction.delay_probability,
                    risk_level=ctx.prediction.risk_level,
                    predicted_delay_days=ctx.prediction.predicted_delay_days,
                    primary_cause_type=ranking.primary_cause_type,
                    selected_cause_types=ranking.selected_cause_types,
                    cause_detail=explanation.cause_detail,
                    analysis_summary=explanation.analysis_summary,
                    recommended_action=explanation.recommended_action,
                    ranking=ranking,
                    due_impact=due,
                    validation=validation,
                    status=WorkflowStatus.REVIEW_REQUIRED,
                    retry_count=retry_count,
                    error_message=reason,
                )

            result = RiskAnalysisResult(
                prediction_id=ctx.prediction.prediction_id,
                order_id=ctx.prediction.order_id,
                plan_id=ctx.prediction.plan_id,
                delay_probability=ctx.prediction.delay_probability,
                risk_level=ctx.prediction.risk_level,
                predicted_delay_days=ctx.prediction.predicted_delay_days,
                primary_cause_type=ranking.primary_cause_type,
                selected_cause_types=ranking.selected_cause_types,
                cause_detail=explanation.cause_detail,
                analysis_summary=explanation.analysis_summary,
                recommended_action=explanation.recommended_action,
                ranking=ranking,
                due_impact=due,
                validation=validation,
                status=WorkflowStatus.COMPLETED,
                retry_count=retry_count,
            )

            self._retry("PersistNode", lambda: self.persist_node.persist(result))
            return result

        except Exception as e:
            self.repository.mark_review_required(prediction_id, str(e))
            return RiskAnalysisResult(
                prediction_id=prediction.prediction_id,
                order_id=prediction.order_id,
                plan_id=prediction.plan_id,
                delay_probability=prediction.delay_probability,
                risk_level=prediction.risk_level,
                predicted_delay_days=prediction.predicted_delay_days,
                primary_cause_type=None,
                selected_cause_types=[],
                cause_detail="상세 리스크 분석 중 오류가 발생했습니다.",
                analysis_summary="상세 분석 실패",
                recommended_action="오류 로그 확인 후 재시도해야 합니다.",
                ranking=None,
                due_impact=None,
                validation=None,
                status=WorkflowStatus.FAILED,
                error_message=str(e),
            )

    def _retry(self, name: str, fn):
        last_error = None
        for attempt in range(self.config.max_retry + 1):
            try:
                return fn()
            except Exception as e:
                last_error = e
                if attempt >= self.config.max_retry:
                    break
        raise RuntimeError(f"{name} failed after retries: {last_error}") from last_error


# =========================
# 11. Notebook용 In-memory Repository
# =========================

class InMemoryRiskAnalysisRepository:
    """
    Notebook에서 로직을 검증하기 위한 저장소입니다.
    실제 운영에서는 이 클래스를 DB Repository로 교체하세요.
    """

    def __init__(self):
        self.predictions: Dict[int, PredictionInput] = {}
        self.contexts: Dict[int, RiskContext] = {}
        self.persisted_results: Dict[int, RiskAnalysisResult] = {}
        self.review_required: Dict[int, str] = {}

    def fetch_non_safe_predictions(self, limit: int = 100) -> List[PredictionInput]:
        rows = [p for p in self.predictions.values() if p.risk_level != RiskLevel.SAFE]
        return rows[:limit]

    def get_prediction(self, prediction_id: int) -> PredictionInput:
        if prediction_id not in self.predictions:
            raise KeyError(f"prediction_id not found: {prediction_id}")
        return self.predictions[prediction_id]

    def load_context(self, prediction_id: int) -> RiskContext:
        if prediction_id not in self.contexts:
            raise KeyError(f"context not found: {prediction_id}")
        return self.contexts[prediction_id]

    def persist_analysis(self, result: RiskAnalysisResult) -> None:
        self.persisted_results[result.prediction_id] = result

    def mark_review_required(self, prediction_id: int, reason: str) -> None:
        self.review_required[prediction_id] = reason


def build_sample_repository() -> InMemoryRiskAnalysisRepository:
    repo = InMemoryRiskAnalysisRepository()

    prediction = PredictionInput(
        prediction_id=10001,
        order_id=123,
        plan_id=456,
        delay_probability=0.72,
        risk_level=RiskLevel.CRITICAL,
        predicted_delay_days=0.5,  # 12시간
    )

    order = OrderInfo(
        order_id=123,
        order_code="ORD-2026-00123",
        product_id=31,
        order_quantity=1200,
        due_at=datetime(2026, 6, 12, 9, 0, 0),
        order_amount=50_000_000,
    )

    plan = PlanInfo(
        plan_id=456,
        order_id=123,
        product_id=31,
        line_id=3,
        planned_start_at=datetime(2026, 6, 11, 9, 0, 0),
        planned_end_at=datetime(2026, 6, 12, 1, 0, 0),
        planned_qty=1200,
        hourly_capacity=100,
    )

    ctx = RiskContext(
        prediction=prediction,
        order=order,
        plan=plan,
        materials=[
            MaterialRequirement(
                material_id="RM-102",
                material_name="RM-102",
                required_qty=1200,
                available_qty=950,
                inbound_qty=0,
                material_required_at=datetime(2026, 6, 11, 9, 0, 0),
                shortage_blocking_hours=12,
            )
        ],
        yield_status=YieldStatus(
            product_id=31,
            line_id=3,
            standard_yield=0.94,
            recent_yield=0.89,
            planned_qty=1200,
            hourly_production_qty=100,
        ),
        machines=[
            MachineStatus(
                machine_id="MCH-03",
                line_id=3,
                status="NORMAL",
                is_core_machine=True,
                has_alternative=False,
            )
        ],
        line_status=LineStatus(
            line_id=3,
            line_name="LINE-03",
            line_status="RUNNING",
            utilization_rate=0.91,
            queue_wait_hours=7,
            actual_throughput=82,
            standard_throughput=100,
            setup_time_hours=6,
            standard_setup_time_hours=3,
            previous_process_delay_hours=9,
        ),
        extra={
            "affected_order_count": 2,
            "expected_loss_amount": 3_200_000,
            "penalty_risk": True,
        },
    )

    repo.predictions[prediction.prediction_id] = prediction
    repo.contexts[prediction.prediction_id] = ctx
    return repo


def result_to_display_text(result: RiskAnalysisResult) -> str:
    cause_name = CAUSE_KOREAN_NAME.get(result.primary_cause_type, "확인 필요") if result.primary_cause_type else "확인 필요"
    secondary = []
    if result.ranking:
        secondary = [
            CAUSE_KOREAN_NAME[c]
            for c in result.ranking.secondary_cause_types
            if c in CAUSE_KOREAN_NAME
        ]

    ranking_evidence = []
    if result.ranking:
        for ranked in result.ranking.priority_ranking:
            ranking_evidence.extend(ranked.evidence)

    lines = [
        f"주문 ID: {result.order_id}",
        f"지연 확률: {result.delay_probability:.2f}",
        f"위험 등급: {result.risk_level.value}",
        f"주요 원인: {cause_name}",
    ]
    if secondary:
        lines.append(f"보조 원인: {', '.join(secondary)}")

    lines.extend([
        "",
        "[상세 원인]",
        result.cause_detail,
        "",
        "[판단 근거]",
    ])

    for i, evidence in enumerate(ranking_evidence[:7], start=1):
        lines.append(f"{i}. {evidence}")

    lines.extend([
        "",
        "[권고 조치]",
        result.recommended_action,
    ])

    if result.validation and result.validation.warnings:
        lines.extend(["", "[검증 경고]"])
        lines.extend([f"- {w}" for w in result.validation.warnings])

    return "\n".join(lines)


Writing risk_analysis_agent_v0.py


## 1. Notebook Smoke Test
샘플 데이터를 이용해 전체 Workflow가 정상 실행되는지 확인합니다.

In [ ]:
from risk_analysis_agent_v0 import (
    build_sample_repository,
    WorkflowController,
    AgentConfig,
    result_to_display_text,
)

repo = build_sample_repository()
controller = WorkflowController(
    repository=repo,
    config=AgentConfig(max_retry=2),
)

result = controller.run_for_prediction(10001)
print(result.status)
print(result_to_display_text(result))


## 2. 저장 결과 확인

`PersistNode`는 실제 DB 대신 `InMemoryRiskAnalysisRepository.persisted_results`에 저장합니다.
실제 운영에서는 이 부분을 DB Repository로 교체합니다.


In [2]:
print("persisted prediction IDs:", list(repo.persisted_results.keys()))
persisted = repo.persisted_results[10001]
print("selected cause types:", [c.value for c in persisted.selected_cause_types])
print("summary:", persisted.analysis_summary)


NameError: name 'repo' is not defined

## 3. 실제 시스템 적용 시 교체해야 하는 부분

### 3-1. DB Repository 구현
`RiskAnalysisRepository` Protocol을 구현해야 합니다.

필수 메서드:
- `fetch_non_safe_predictions`
- `get_prediction`
- `load_context`
- `persist_analysis`
- `mark_review_required`

### 3-2. LLM 구현
현재는 `TemplateExplanationGenerator`가 deterministic 설명을 생성합니다.  
운영에서는 `ExplanationGenerator` Protocol을 구현하여 사내 LLM 또는 OpenAI/Azure/Bedrock 등으로 교체합니다.

### 3-3. Threshold 설정
아래 값은 공정/운영 기준 확정 후 `AgentConfig`에 주입하세요.
- `queue_wait_threshold_hours`
- `setup_overrun_threshold_hours`
- `changeover_overrun_threshold_hours`
- `due_slack_safe_threshold_hours`


In [ ]:
# 예: 운영 threshold를 주입하는 형태
config = AgentConfig(
    max_retry=2,
    line_utilization_threshold=0.85,
    throughput_min_ratio=0.90,
    yield_gap_major_threshold=0.05,
    queue_wait_threshold_hours=None,  # TODO: 운영 기준 확정 필요
    setup_overrun_threshold_hours=None,  # TODO: 운영 기준 확정 필요
    changeover_overrun_threshold_hours=None,  # TODO: 운영 기준 확정 필요
    due_slack_safe_threshold_hours=None,  # TODO: 운영 기준 확정 필요
)
config
